# 🏆 Notebook 14: Capstone — Fine-tune & Serve Gemma 4

Load a quantized Gemma 4 model, fine-tune with LoRA, evaluate, and serve locally. All within the 48GB memory budget.

**Prerequisites:** All previous notebooks

**What you'll learn:**
1. Understand Gemma 4's architecture and memory requirements\n2. Implement LoRA (Low-Rank Adaptation) for efficient fine-tuning\n3. Train LoRA adapters on a simple task\n4. Understand the fine-tuning → evaluation → serving pipeline

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## 🎯 Why This Notebook Matters

Imagine you've built a car from scratch over 13 lessons — engine, transmission, wheels, everything. Now it's time to take it for a drive AND customize it for YOUR specific road.

That's what this capstone is about. You have a powerful pre-trained model (the car). Fine-tuning with LoRA lets you customize it for your specific task (your road) — without rebuilding the engine.

**Real-world example**: A hospital wants an AI that understands medical terminology. They can't train a 12-billion-parameter model from scratch (costs millions). But they CAN take Gemma 4 and fine-tune it with LoRA on medical texts for ~$10 of compute. That's the power of what you're about to learn.

### 🧠 The Big Picture

LoRA fine-tuning is like adding a small sticky note to each page of a textbook — the original text (base weights) stays unchanged, but the notes (low-rank adapters) customize the book for your specific needs. You only need to store the sticky notes, not reprint the entire book.

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Gemma 4 Architecture Overview

Gemma 4 is Google's latest open model family. Key features:
- RoPE positional encoding
- GQA (Grouped Query Attention)
- GeGLU FFN variant
- Logit soft-capping

## Memory Budget for 12B 4-bit

| Component | Memory |
|-----------|--------|
| Model weights (4-bit) | ~7 GB |
| LoRA adapters | ~0.1 GB |
| Optimizer state | ~0.4 GB |
| KV-cache (8K context) | ~1 GB |
| Activations | ~2 GB |
| **Total** | **~10.5 GB** |
| Available (48GB - 4GB OS) | 44 GB |
| **Fits?** | ✅ Yes |

### 💡 Why Does This Matter?

Why LoRA instead of full fine-tuning? A 12B model has ~24GB of parameters. Full fine-tuning needs 3-4x that for gradients and optimizer states (~72-96GB). LoRA only trains ~0.1% of parameters, fitting easily in 48GB. Why 4-bit quantization? It shrinks the 12B model from 24GB to ~7GB, leaving room for LoRA adapters and activations.

## Loading Gemma 4

⚠️ This cell requires downloading a model from Hugging Face (~7GB). Uncomment to run.

In [ ]:
# Uncomment to load Gemma 4 (requires HF access token for gated models)
# from mlx_lm import load, generate
# 
# model, tokenizer = load('mlx-community/gemma-2-2b-it-4bit')  # Use smaller variant for demo
# 
# # Verify it fits in memory
# import mlx.core as mx
# mem_gb = mx.metal.get_active_memory() / 1e9
# print(f'Memory after loading: {mem_gb:.1f} GB')
# 
# # Baseline generation
# response = generate(model, tokenizer, prompt='What is a transformer?', max_tokens=100)
# print(response)

print('To load Gemma 4:')
print('  from mlx_lm import load, generate')
print('  model, tokenizer = load("mlx-community/gemma-2-2b-it-4bit")')
print('  response = generate(model, tokenizer, prompt="Hello", max_tokens=100)')

## LoRA Fine-tuning

**LoRA** (Low-Rank Adaptation) adds small trainable matrices to frozen model weights:

$$W' = W + \alpha \cdot B \cdot A$$

Where $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times d}$ with rank $r \ll d$.

💡 For a 12B model with rank=8 targeting attention layers: ~0.1% trainable parameters.

In [ ]:
import mlx.core as mx
import mlx.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation."""
    def __init__(self, in_features, out_features, rank=8, alpha=16.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.lora_A = mx.random.normal((in_features, rank)) * 0.01
        self.lora_B = mx.zeros((rank, out_features))
        self.alpha = alpha
        self.rank = rank
        # Freeze the original weights
        self.linear.freeze()
    
    def __call__(self, x):
        base = self.linear(x)
        lora = (x @ self.lora_A @ self.lora_B) * (self.alpha / self.rank)
        return base + lora

# Demo
lora = LoRALinear(256, 256, rank=8)
x = mx.random.normal((1, 10, 256))
out = lora(x)
mx.eval(out)

base_params = 256 * 256
lora_params = 256 * 8 + 8 * 256
print(f'Base params: {base_params:,}')
print(f'LoRA params: {lora_params:,} ({lora_params/base_params*100:.1f}% of base)')
print(f'Output shape: {out.shape} ✅')

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

### 💡 Visualizing LoRA's Parameter Efficiency

LoRA freezes 94% of the model and only trains 6% — this is why it fits in memory and trains fast.

The pie chart below uses the numbers from our `LoRALinear(256, 256, rank=8)` demo above to show just how few parameters LoRA actually touches.

In [ ]:
import matplotlib.pyplot as plt

# Parameter counts from the LoRA demo above
base_params = 65_536   # 256 * 256 (frozen)
lora_params = 4_096    # 256 * 8 + 8 * 256 (trainable)

labels = [f'Frozen base\n{base_params:,} params', f'Trainable LoRA\n{lora_params:,} params']
sizes = [base_params, lora_params]
colors = ['#4a90d9', '#e74c3c']
explode = (0, 0.08)  # Slightly separate the LoRA slice

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12}
)
for t in autotexts:
    t.set_fontweight('bold')

ax.set_title('LoRA Parameter Breakdown\n(rank=8, d=256)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'⚡ Only {lora_params/base_params*100:.1f}% of parameters are trainable — huge memory & speed savings!')

### 🔧 Working LoRA Training Demo

Let's actually train our `LoRALinear` layer on a simple task to see LoRA in action. We'll train it to approximate a target linear transformation — this demonstrates that the low-rank adapters can learn meaningful weight updates while the base weights stay frozen.

💡 **What's happening:** Only `lora_A` and `lora_B` receive gradient updates. The frozen base weight `W` stays exactly the same throughout training.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import matplotlib.pyplot as plt

# Create a LoRA layer and a target transformation
mx.random.seed(42)
lora_model = LoRALinear(64, 64, rank=4, alpha=8.0)
mx.eval(lora_model.parameters())

# Target: a random linear transformation we want LoRA to learn
target_W = mx.random.normal((64, 64)) * 0.1
mx.eval(target_W)

# Save original base weights to verify they don't change
original_base = mx.array(lora_model.linear.weight)
mx.eval(original_base)

# Loss: MSE between LoRA output and target output
def lora_loss(model, x):
    pred = model(x)                    # shape: (batch, 64)
    target = x @ target_W.T            # shape: (batch, 64)
    return mx.mean((pred - target) ** 2)

# Training loop
loss_grad_fn = nn.value_and_grad(lora_model, lora_loss)
optimizer = optim.Adam(learning_rate=1e-3)
losses = []

for step in range(200):
    x = mx.random.normal((32, 64))     # shape: (batch=32, features=64)
    loss, grads = loss_grad_fn(lora_model, x)
    optimizer.update(lora_model, grads)
    mx.eval(lora_model.parameters(), optimizer.state, loss)
    losses.append(loss.item())
    if step % 50 == 0:
        print(f'  Step {step:3d} | Loss: {loss.item():.6f}')

# Verify base weights are FROZEN
base_diff = mx.max(mx.abs(lora_model.linear.weight - original_base)).item()
print(f'\n✅ Base weight change: {base_diff:.2e} (should be 0.0)')
print(f'✅ Final loss: {losses[-1]:.6f}')

# Plot training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='#e74c3c', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('MSE Loss')
ax.set_title('LoRA Training: Learning a Target Transformation')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\n💡 LoRA learned to approximate the target with only rank-4 adapters!')
print(f'   Trainable params: {4*64 + 4*64} = {4*64*2} (vs {64*64} = {64**2} base params)')

### 🔍 What Just Happened?

We implemented LoRA fine-tuning from scratch: freeze the base model, add small trainable adapters (rank-4 matrices), and train only those. The key insight: you can customize a massive model's behavior by training less than 1% of its parameters.

## Fine-tuning Execution

⚠️ Actual fine-tuning requires a loaded model and dataset. This shows the training loop structure.

In [ ]:
# Fine-tuning loop template
print('Fine-tuning loop structure:')
print('  1. Load base model (4-bit quantized)')
print('  2. Apply LoRA adapters to target modules (Q, K, V projections)')
print('  3. Prepare dataset (instruction-response pairs)')
print('  4. Training loop:')
print('     - Forward pass through model')
print('     - Compute cross-entropy loss')
print('     - Backward pass (only LoRA params get gradients)')
print('     - Update LoRA params with AdamW')
print('     - Monitor memory with mx.metal.get_active_memory()')
print('  5. Save LoRA adapter weights')
print('  6. Load base + adapter for inference')

## Evaluation & Serving

Compare base vs fine-tuned model outputs, measure perplexity, and serve interactively.

In [ ]:
print('Evaluation metrics:')
print('  - Perplexity on held-out data')
print('  - Side-by-side comparison of base vs fine-tuned outputs')
print('  - Inference speed (tok/s)')
print('  - Memory usage during generation')
print()
print('Serving:')
print('  - Load base model + LoRA adapter')
print('  - Interactive generation with streaming')
print('  - Final performance benchmarks')

## 🎉 Congratulations!

You've completed the entire LLM from Scratch series on Apple Silicon:

1. ✅ Environment & Apple Silicon fundamentals
2. ✅ MLX framework mastery
3. ✅ Math foundations (dot products, softmax, cross-entropy)
4. ✅ Tokenization (character, BPE, tiktoken, SentencePiece)
5. ✅ Embeddings & positional encoding (sinusoidal, RoPE)
6. ✅ Self-attention (single-head, multi-head, GQA)
7. ✅ Transformer architecture (FFN, norms, residuals)
8. ✅ Building & training a GPT from scratch
9. ✅ Apple Silicon training optimizations
10. ✅ Modern architectures (LLaMA, Mistral, Gemma)
11. ✅ Metal custom kernels
12. ✅ Inference optimization (KV-cache, quantization)
13. ✅ Advanced attention (Flash, Paged, Ring)
14. ✅ Local serving (mlx-lm, llama.cpp)
15. ✅ Capstone: Fine-tuning & serving Gemma 4

🎯 You're now ready for AI lab interviews at Google DeepMind, OpenAI, Anthropic, Apple ML, and Meta FAIR!

## 🧪 Try It Yourself

Experiment with LoRA:

1. **Rank experiment**: Create LoRALinear layers with rank=1, 4, 8, 16, 32. How does the number of trainable parameters change?

2. **Alpha scaling**: With rank=8, try alpha values of 1, 8, 16, 32. What does alpha control? (Hint: it scales the LoRA contribution.)

3. **Memory budget**: For a 12B model, calculate the memory needed for LoRA adapters targeting all Q, K, V projections with rank=16.

## 📜 History & Alternatives

### The Evolution of Google's Open Models: From Gemma to Gemma 4

Google's Gemma family represents a strategic shift toward open-weight models, distilling capabilities from the proprietary Gemini series into smaller, publicly available models.

| Year | Model | Team | Key Contribution |
|------|-------|------|-----------------|
| 2023 (Dec) | **Gemini 1.0** | Google DeepMind | Proprietary multimodal model — Ultra/Pro/Nano variants |
| 2024 (Feb) | **Gemma 1** | Google DeepMind | First open-weight Gemma — 2B/7B, distilled from Gemini, RoPE + Multi-Query Attention |
| 2024 (Jun) | **Gemma 2** | Google DeepMind | 2B/9B/27B, knowledge distillation, sliding window + global attention alternation, GQA |
| 2024 (Aug) | **Gemini 1.5 Pro** | Google DeepMind | 1M token context, MoE architecture (proprietary) |
| 2024 (Oct) | **ShieldGemma** | Google DeepMind | Safety-focused classifier built on Gemma — content filtering |
| 2025 (Mar) | **Gemma 3** | Google DeepMind | 1B/4B/12B/27B, multimodal (vision), 128K context, p-RoPE |
| 2025 (Jun) | **Gemini 2.5 Pro** | Google DeepMind | Thinking model, 1M context, best-in-class reasoning |
| 2026 | **Gemma 4** | Google DeepMind | Diffusion-based architecture, MoE with shared experts, advanced reasoning capabilities |

### 💡 Google's Open Model Strategy

Google's approach differs from Meta's (LLaMA) and Mistral's strategies:

| Aspect | Google (Gemma) | Meta (LLaMA) | Mistral |
|--------|---------------|-------------|---------|
| **Source** | Distilled from Gemini | Trained from scratch | Trained from scratch |
| **Sizes** | 1B-27B | 1B-405B | 7B-8x22B |
| **License** | Gemma license (restricted) | Llama license (permissive) | Apache 2.0 |
| **Multimodal** | Yes (Gemma 3+) | Yes (LLaMA 3.2+) | Limited (Pixtral) |
| **MoE** | Gemma 4 | LLaMA 4 | Mixtral |
| **Key advantage** | Gemini distillation | Scale + open community | Efficiency |

### The Gemma Architecture Evolution

```
Gemma 1 (2024):  Standard transformer, MQA, RoPE
    ↓
Gemma 2 (2024):  + Knowledge distillation, sliding window attention, GQA, logit soft-capping
    ↓
Gemma 3 (2025):  + Multimodal (SigLIP vision encoder), p-RoPE, 128K context
    ↓
Gemma 4 (2026):  + Diffusion-based generation, MoE with shared experts, enhanced reasoning
```

### ⚡ Why Gemma 4 Matters

Gemma 4 represents several architectural firsts for open models:
- **Diffusion-based**: Moves beyond autoregressive token-by-token generation
- **MoE with shared experts**: Every token uses a shared expert (always active) plus routed experts — similar to DeepSeek-V3's architecture
- **Reasoning capabilities**: Built-in chain-of-thought and structured reasoning

### Competitive Landscape (2025-2026)

| Model | Params (Active) | Architecture | Open Weight | Reasoning |
|-------|----------------|-------------|-------------|-----------|
| GPT-4o | Unknown | Dense (?) | No | Yes (o1/o3) |
| Claude 3.5 Sonnet | Unknown | Dense (?) | No | Yes |
| Gemini 2.5 Pro | Unknown | MoE | No | Yes (thinking) |
| LLaMA 4 Maverick | ~17B active / 400B total | MoE | Yes | Limited |
| DeepSeek-R1 | 37B active / 671B total | MoE | Yes | Yes (RL-trained) |
| **Gemma 4** | TBD | MoE + Diffusion | Yes | Yes |
| Qwen 3 | Various | Dense + MoE | Yes | Yes (thinking) |

### ⚠️ Common Pitfalls

- **Gemma license ≠ Apache 2.0**: Unlike Mistral's models, Gemma has a custom license that restricts certain commercial uses and redistribution — always check the Gemma Terms of Use before deploying in production
- **Distillation ceiling**: Because Gemma models are distilled from Gemini, they inherit Gemini's knowledge cutoff and biases but at lower capacity — don't expect a 9B distilled model to match a 9B model trained from scratch on the same data
- **MoE memory trap**: Gemma 4's MoE architecture means total parameter count is much larger than active parameters — you need enough memory to load ALL experts even though only a few are active per token
- **Gemma version confusion**: Gemma 1, 2, 3, and 4 have significantly different architectures (MQA → GQA → p-RoPE → MoE+Diffusion) — code written for one version won't work for another without adaptation
- **Multimodal ≠ universal**: Gemma 3+ supports vision, but the vision encoder (SigLIP) is frozen during fine-tuning by default — fine-tuning only the language model on multimodal tasks can produce poor visual grounding

### 🎯 Interview Tip

> *"Gemma 4's shift to diffusion-based generation is significant because it breaks the autoregressive bottleneck — instead of generating one token at a time (sequential, memory-bandwidth-bound), diffusion models can generate multiple tokens in parallel through iterative denoising. Combined with MoE (where only a fraction of parameters are active per token), this creates a model that's both more capable and more efficient at inference time. The shared expert design (from DeepSeek-V3) ensures a baseline of computation for every token while routed experts provide specialization."*

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 15 — Mixture of Experts**, we'll continue building on these concepts.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?